# 基于 ResNet 的 K 线图涨跌趋势分类研究\n## 魔搭社区 (ModelScope) GPU 实验\n\n### 运行前准备\n1. 将 `all_ohlc_data.pkl` 上传到 `/mnt/workspace/data/raw/`\n2. 将 `src/` 目录下全部 `.py` 文件放到 `/mnt/workspace/src/`\n3. 按顺序运行下方 Cell\n\n**环境**: Ubuntu 22.04 | CUDA 12.8 | PyTorch 2.10 | 24GB GPU

## Cell 0: 环境检查

In [ ]:
import os, sys, subprocess

# 确保在 workspace 目录
os.chdir('/mnt/workspace')
os.environ['MODELSCOPE_BASE_DIR'] = '/mnt/workspace'
sys.path.insert(0, '/mnt/workspace/src')

# 检查 GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.0f} GB')

# 检查文件
print(f'\n工作目录: {os.getcwd()}')
!ls -lh /mnt/workspace/data/raw/ 2>/dev/null || echo 'data/raw 目录为空'
!ls /mnt/workspace/src/ 2>/dev/null || echo 'src 目录为空'

## Cell 1: 安装依赖

In [ ]:
# 安装所需包（魔搭预装了 torch/torchvision，这里补装其余依赖）
!pip install mplfinance scikit-learn tqdm akshare -i https://mirrors.aliyun.com/pypi/simple/ --trusted-host mirrors.aliyun.com -q
print('依赖安装完成')

## Cell 2: 加载数据 (Phase 1)

In [ ]:
import pickle
from config import *

data_path = os.path.join(RAW_DATA_DIR, 'all_ohlc_data.pkl')

if os.path.exists(data_path):
    with open(data_path, 'rb') as f:
        all_data = pickle.load(f)
    print(f'数据加载成功: {len(all_data)} 只股票')
    
    # 统计
    total_samples = sum(len(df) for df in all_data.values())
    print(f'总样本数: {total_samples}')
    
    # 检查列名
    sample_code = list(all_data.keys())[0]
    print(f'列名: {all_data[sample_code].columns.tolist()}')
else:
    print(f'数据文件不存在: {data_path}')
    print('请将 all_ohlc_data.pkl 上传到 /mnt/workspace/data/raw/')
    print()
    print('=== 备用方案: 从 AKShare 重新获取数据 ===')
    %run /mnt/workspace/src/01_fetch_data.py

## Cell 3: 渲染 K 线图 (Phase 2)

In [ ]:
%%time
# 渲染 K 线图 (GPU 环境 STRIDE=15, ~26k 张)
# 使用多进程加速
%run /mnt/workspace/src/02_render_charts.py

## Cell 4: 验证 DataLoader (Phase 3)

In [ ]:
from dataset import get_dataloaders

train_loader, val_loader, test_loader = get_dataloaders()
print(f'Train batches: {len(train_loader)}')
print(f'Val batches: {len(val_loader)}')
print(f'Test batches: {len(test_loader)}')

# 验证一个 batch
imgs, labels = next(iter(train_loader))
print(f'Input shape: {imgs.shape}')   # [128, 3, 224, 224]
print(f'Label shape: {labels.shape}')  # [128]
print(f'Label distribution: up={(labels==1).sum()}, down={(labels==0).sum()}')

## Cell 5: 构建模型 (Phase 4)

In [ ]:
from model import build_model

model = build_model()
print(f'Model loaded on {DEVICE}')

# 显存预估
mem_params = sum(p.numel() * p.element_size() for p in model.parameters()) / 1024**2
print(f'参数内存: {mem_params:.1f} MB')
print(f'Batch 128 估计总显存: ~{mem_params*3 + 500:.0f} MB (含梯度+激活)')

## Cell 6: 训练 (Phase 5)

In [ ]:
%%time
# GPU 训练，预计 15-20 分钟完成 25 epochs
%run /mnt/workspace/src/train.py

## Cell 7: 测试集评估 (Phase 6)

In [ ]:
%run /mnt/workspace/src/evaluate.py

## Cell 8: 结果展示

In [ ]:
import pickle
from IPython.display import Image, display
from config import OUTPUT_DIR

# 显示结果文本
results_path = f'{OUTPUT_DIR}/results.txt'
if os.path.exists(results_path):
    with open(results_path, 'r', encoding='utf-8') as f:
        print(f.read())

# 显示混淆矩阵
cm_path = f'{OUTPUT_DIR}/confusion_matrix.png'
if os.path.exists(cm_path):
    display(Image(filename=cm_path))

# 显示 ROC 曲线
roc_path = f'{OUTPUT_DIR}/roc_curve.png'
if os.path.exists(roc_path):
    display(Image(filename=roc_path))

# 显示训练曲线
train_curve_path = f'{OUTPUT_DIR}/training_curves.png'
if os.path.exists(train_curve_path):
    display(Image(filename=train_curve_path))

## Cell 9: 保存模型文件（可选）

In [ ]:
# 列出所有输出文件
import os
print('=== 输出文件 ===')
for root, dirs, files in os.walk(OUTPUT_DIR):
    for f in files:
        path = os.path.join(root, f)
        size_mb = os.path.getsize(path) / 1024**2
        print(f'{size_mb:5.1f} MB  {path}')

print()
print('=== 图片统计 ===')
for split in ['train', 'val', 'test']:
    for label in ['up', 'down']:
        d = f'{IMAGE_DIR}/{split}/{label}'
        if os.path.exists(d):
            count = len([f for f in os.listdir(d) if f.endswith('.png')])
            print(f'  {split}/{label}: {count}')